#### Rainfall-Runoff simulation using LSTMs

This notebook serves as an interactive hands-on practical exercise accompanying the lecture **"Rainfall-runoff simulation using LSTMs"**, offered as part of the ECMWF training series on **Machine Learning for Earth Systems Modelling**.

The objective of this tutorial is to guide you through the complete end-to-end workflow of building, training, evaluating, and analyzing a deep learning model for rainfall-runoff simulation using a Long Short-Term Memory (LSTM) model.

Tools and data

- Model Framework: Built using the [Hy2DL Library](https://github.com/eduardoAcunaEspinoza/Hy2DL), an open-source Python toolkit designed specifically for deep learning applications in hydrology
- Dataset: A regional subset of [50 hydrological](https://zenodo.org/records/22085206) basins across South-West Germany extracted from the CAMELS-DE dataset ([Read paper / DOI](https://doi.org/10.5194/essd-16-5625-2024))

Authors
- Eduardo Acuña (eduardo.espinoza@kit.edu)
- Ralf Loritz
- Uwe Ehret

In [ ]:
import os

COLAB_ROOT = "/content"
REPO_NAME = "DeepLearningHydrology-Course"
REPO_PATH = os.path.join(COLAB_ROOT, REPO_NAME)
os.chdir(COLAB_ROOT)

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/eduardoAcunaEspinoza/DeepLearningHydrology-Course.git

os.chdir(REPO_PATH)
print("Current working directory:", os.getcwd())

# Install the Hy2DL library using uv
print("Installing Hy2DL and dependencies with uv...")
!pip install uv --quiet
!uv pip install hy2dl --system

# Download subset of CAMELS-DE from Zenodo
if not os.path.exists("./data/CAMELS-DE"):
    print("Downloading CAMELS-DE subset from Zenodo...")
    !wget -O camels_de.zip "https://zenodo.org/records/22085206/files/CAMELS-DE-subset.zip?download=1" --quiet
    !unzip -q -o camels_de.zip -d ./data/
    !rm camels_de.zip # Clean up the zip file to save space

print("Setup complete and dataset is ready!")

In [ ]:
import datetime
import inspect
import logging
import os
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr
from hy2dl.datasetzoo import get_dataset
from hy2dl.evaluation import calculate_metrics, get_tester
from hy2dl.modelzoo import get_model
from hy2dl.training.basetrainer import BaseTrainer
from hy2dl.utils.config import Config

# Suppress unnecesary warnings due to distributed loggers
logging.getLogger("distributed").setLevel(logging.ERROR)

base_dir = Path.cwd().resolve()
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}

COLAB_ROOT = "/content"
REPO_NAME = "DeepLearningHydrology-Course"
REPO_PATH = os.path.join(COLAB_ROOT, REPO_NAME)
os.chdir(REPO_PATH)

##### Part 1: Initialize experiment settings

In this step, we load the experiment configuration from a YAML file (`camels_de.yml`). The `Config` class handles parsing key hyperparameters and environment options, including:

- Time spans: Start and end dates for training, validation, and testing periods.
- Features: Selection of meteorological input forcings, static attributes and target variables.
- Model hyperparameters: LSTM hidden state dimensions, sequence length, batch size, learning rate, epochs...

In [ ]:
# Path to .yml file where the experiment settings are stored.
path_experiment_settings = "examples/configs/camels_de.yml"

# Read experiment settings
config = Config(path_experiment_settings, base_dir=base_dir)
config.init_experiment()
config.dump()

Dataset = get_dataset(config)
Tester = get_tester(config)

##### Part 2: Prepare training and validation data

Here, we initialize the data pipelines for both the training and validation splits. In both cases we process the basin-wise time series information and static attributes, and assemble them together in a dataset object, that Pytorch's dataloaders will use to construct the batches. The training object contains the model and the pipeline to train it, while the validation tester focuss on evaluating model performance. 

In [ ]:
# Create training dataset
training_dataset = Dataset(cfg=config, time_period="training")
training_dataset.setup_dataset()
# Initialize training object
trainer = BaseTrainer(cfg=config, training_dataset=training_dataset)

In [ ]:
validation_dataset = Dataset(cfg=config, time_period="validation")
validation_dataset.setup_dataset(check_nan=False, path_scaler = config.path_save_folder / "scaler.yml" )
tester_validation = Tester(cfg=config, evaluation_dataset=validation_dataset)

##### Part 3: Train the LSTM model

This section executes the main training loop:

- For each epoch, the model performs forward passes over all batches, computes the loss and updates weights via backpropagation using the ADAM algorithm.
- At the end of the epoch, the model is evaluated on the validation dataset to log validation performance metrics.
- The model's weights are saved after each epoch

In [ ]:
# Training report structure
validation_headers = "".join([f"{m:^10}|" for m in config.validation_metric])
config.logger.info("Training model".center(60, "-"))
config.logger.info(f"{'':^16}|{'Training':^21}|{'Validation':^{(11 * len(config.validation_metric)) + 10}}|")
config.logger.info(f"{'Epoch':^5}|{'LR':^10}|{'Loss':^10}|{'Time':^10}|{validation_headers}{'Time':^10}|")

# Loop through epochs
total_time = time.time()
for epoch in range(1, config.epochs + 1):
    trainer.train_model(epoch=epoch)  # Training
    tester_validation.validate_model(model=trainer.model, epoch=epoch)  # Validation
    config.logger.info(trainer.report + tester_validation.validation_report)  # report

config.logger.info(f"Total training time: {datetime.timedelta(seconds=int(time.time() - total_time))}\n")
shutil.rmtree(tester_validation.path_zarr, ignore_errors=True)  # delete validation results

##### Part 4: Evaluate model on test dataset

Once training is complete, we evaluate the trained model on unseen test data. For this we:

* Load the trained model weights saved from the final epoch (or best epoch).
* Construct the test dataset
* Run inference on the test dataset
* Store the predicted discharge (`y_sim`) and actual observed discharge (`y_obs`) into an efficient Zarr dataset format for evaluation.

In [ ]:
# If I already trained a model, I can re-construct it using the saved parameters from a given epoch
model = get_model(config).to(config.device)
model.load_state_dict(torch.load(config.path_save_folder / "model" / f"model_epoch_{config.epochs}", map_location=config.device))

In [ ]:
testing_dataset = Dataset(cfg=config, time_period="testing")
testing_dataset.setup_dataset(check_nan=False, path_scaler = config.path_save_folder / "scaler.yml" )
tester_testing = Tester(cfg=config, evaluation_dataset=testing_dataset)

config.logger.info("Testing model...")
testing_time = time.time()
tester_testing.evaluate_model(model = trainer.model)
config.logger.info("Testing completed.")
config.logger.info(f"Total testing time: {datetime.timedelta(seconds=int(time.time() - testing_time))}\n")

##### Part 5: Initial analysis and visualization

In this final step, we analyze model performance:

* Performance metrics: Compute metric scores across all test basins and plot their distribution.
* Hydrograph visualization: Plot observed versus predicted river discharge for a single randomly selected basin and year for visual inspection.

In [ ]:
test_results = xr.open_zarr(tester_testing.path_zarr)
testing_metrics = calculate_metrics(ds_results=test_results, metric_name = config.testing_metrics)
testing_metrics.to_zarr(config.path_save_folder / "testing_metrics.zarr", mode="w")

In [ ]:
# Loss testing
target_of_interest = random.sample(list(testing_metrics.feature.values), 1)[0]
test_metric = testing_metrics.sel(feature=target_of_interest, metric="nse").round(3).T.to_pandas().dropna()
# Plot the histogram
plt.figure(figsize=(10, 5))
plt.hist(test_metric, bins = np.linspace(0.0, 1.0, 11).tolist())
# Add NSE statistics to the plot
plt.text(
    0.01,
    0.8,
    (
        f"Mean: {'%.2f' % test_metric.mean():>7}\n"
        f"Median: {'%.2f' % test_metric.median():>0}\n"
        f"Max: {'%.2f' % test_metric.max():>9}\n"
        f"Min: {'%.2f' % test_metric.min():>10}"
    ),
    transform=plt.gca().transAxes,
    bbox=dict(facecolor="white", alpha=0.5),
)

# Format plot
plt.xlabel("NSE", fontsize=12, fontweight="bold")
plt.ylabel("Frequency", fontsize=12, fontweight="bold")
plt.title(f"NSE histogram for: {target_of_interest}", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Plot simulated and observed discharges
basin_to_analyze = random.sample(list(test_results.gauge_id.values), 1)[0]
y_sim = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_sim"].compute()
y_obs = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_obs"].compute()

# Select a random year available in the date coordinate
available_years = np.unique(y_obs.date.dt.year)
random_year = str(np.random.choice(available_years))

# Select year subset
y_sim_year = y_sim.sel(date=random_year)
y_obs_year = y_obs.sel(date=random_year)

plt.figure(figsize=(15, 7.5))
plt.plot(y_obs_year.date, y_obs_year, label="observed", color=color_palette["observed"])
plt.plot(y_sim_year.date, y_sim_year, label="simulated", alpha=0.5, color=color_palette["simulated"])

# Format plot
plt.xlabel("Date", fontsize=16, fontweight="bold")
plt.ylabel(target_of_interest, fontsize=16, fontweight="bold")
plt.title(f"Results for gauge_id {basin_to_analyze}, year {random_year}", fontsize=20, fontweight="bold")
plt.tick_params(axis="both", which="major", labelsize=12)
plt.legend(loc="upper right", fontsize=16)
plt.tight_layout()
plt.show()